# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vishalseries7030/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I will use a Logistic Regression classifier as the first interpretable machine-learning model. The target will be a proxy label derived from the existing trend_direction field, with declining content represented as 1 and other observed directions as 0. The label-source fields trend_direction and trend_pct will not be used as model features. I will use a client-grouped train/test split to reduce the risk of the same client's content appearing in both sets. This experiment measures agreement with the dataset's derived trend label; it does not prove that a page refresh will improve performance.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will use a client-grouped train/test split so that content from the same client does not appear in both sets. This tests whether the model's classification of the existing trend proxy generalizes to held-out clients. The target is derived from observed trend data, so this is not a future-outcome forecast. Grouping by client reduces client-specific overlap, but it does not by itself eliminate temporal or label-construction leakage. I will inspect the target definition and exclude label-source fields from the model features before interpreting results.

In [2]:

# ============================================================
# ML-08 | SECTION 2: SPLIT DESIGN
# Complete code: download repo, locate dataset, create grouped split
# ============================================================

import subprocess
import pandas as pd
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

# 1. Locate the repository in Colab or clone it if needed
REPO_URL = "https://github.com/vishalseries7030/flyrank-ml-internship.git"
REPO_DIR = Path("/content/flyrank-ml-internship")

if not REPO_DIR.exists():
    result = subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        raise RuntimeError(
            "Could not clone the repository. Check the GitHub URL and "
            "Colab internet access.\n" + result.stderr[-2000:]
        )

print("Repository:", REPO_DIR)

# 2. Find the dataset file
matches = list(REPO_DIR.rglob("content_refresh_anonymized.csv"))

if not matches:
    raise FileNotFoundError(
        "Dataset file 'content_refresh_anonymized.csv' was not found "
        "inside the cloned repository."
    )

DATA_PATH = matches[0]
print("Dataset found:", DATA_PATH)

# 3. Load and validate data
df = pd.read_csv(DATA_PATH)

required = {"content_id", "client_id", "trend_direction"}
missing_cols = required - set(df.columns)

if missing_cols:
    raise ValueError(f"Missing required columns: {sorted(missing_cols)}")

print("\nDataset shape:", df.shape)
print("\nTrend direction values:")
print(df["trend_direction"].value_counts(dropna=False).to_string())

# 4. Create an explicit proxy target from the existing trend label
direction = df["trend_direction"].astype("string").str.strip().str.lower()

decline_values = {
    "down", "declining", "decline", "decreasing", "negative"
}

observed_values = set(direction.dropna().unique())
matched_values = observed_values & decline_values

if not matched_values:
    raise ValueError(
        "No recognised declining label was found. Review the printed "
        "trend_direction values and update decline_values to match the "
        "dataset's actual category. Do not guess the label."
    )

valid = direction.notna() & df["client_id"].notna()

model_df = df.loc[valid].copy()
model_df["target_declining_proxy"] = (
    direction.loc[valid].isin(decline_values).astype("int8")
)

if model_df["target_declining_proxy"].nunique() != 2:
    raise ValueError(
        "The target contains only one class after filtering. "
        "Inspect the label values before proceeding."
    )

print("\nDeclining category value(s) used:", sorted(matched_values))
print("Rows with observed labels and client IDs:", len(model_df))
print(
    "Proxy target counts:\n",
    model_df["target_declining_proxy"].value_counts().sort_index().to_string()
)

# 5. Grouped split: a client must not occur in both sets
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        model_df,
        y=model_df["target_declining_proxy"],
        groups=model_df["client_id"]
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

train_clients = set(train_df["client_id"])
test_clients = set(test_df["client_id"])
client_overlap = train_clients & test_clients

print("\n--- SPLIT SUMMARY ---")
print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(client_overlap))

print("\nTraining target distribution:")
print(train_df["target_declining_proxy"].value_counts(normalize=True).sort_index().round(3))

print("\nTest target distribution:")
print(test_df["target_declining_proxy"].value_counts(normalize=True).sort_index().round(3))

assert len(client_overlap) == 0, "ERROR: client overlap detected!"
print("\nPASS: no client appears in both train and test.")

print(
    "\nInterpretation: this is a client-grouped test of an existing "
    "trend-label proxy, not a forecast of future decline. The split "
    "does not by itself eliminate temporal or label-construction leakage."
)


Repository: /content/flyrank-ml-internship
Dataset found: /content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv

Dataset shape: (30000, 44)

Trend direction values:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152

Declining category value(s) used: ['down']
Rows with observed labels and client IDs: 30000
Proxy target counts:
 target_declining_proxy
0    13738
1    16262

--- SPLIT SUMMARY ---
Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Client overlap: 0

Training target distribution:
target_declining_proxy
0    0.45
1    0.55
Name: proportion, dtype: float64

Test target distribution:
target_declining_proxy
0    0.489
1    0.511
Name: proportion, dtype: float64

PASS: no client appears in both train and test.

Interpretation: this is a client-grouped test of an existing trend-label proxy, not a forecast of future decline. The split does not by itself eliminate temporal or label-construction le

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I will compare the rule-based ML-07 review score and a Logistic Regression classifier on the same client-held-out test set. Both rankings will be evaluated against the same existing proxy label, where trend_direction == "down" is positive. I will report the test-set positive rate, Average Precision, ROC-AUC, and Precision@20 for both approaches. The baseline score was designed for review prioritization, not decline classification, so this comparison is exploratory rather than proof that one approach is universally better. Because the target is derived from observed trend data and features may share the same observation period, the results must not be interpreted as a forecast of future decline or as evidence that content refreshes cause improvements.

In [3]:

# ============================================================
# ML-08 | SECTION 3: TRAIN + COMPARE AGAINST ML-07 BASELINE
# Same client-held-out test set; same proxy target; same metrics
# ============================================================

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score
)

# 1. Confirm Section 2 variables are available
if "train_df" not in globals() or "test_df" not in globals():
    raise RuntimeError(
        "Run Section 2 first so train_df and test_df exist."
    )

if "target_declining_proxy" not in train_df.columns:
    raise ValueError("Proxy target is missing from train_df.")

if "target_declining_proxy" not in test_df.columns:
    raise ValueError("Proxy target is missing from test_df.")

train = train_df.copy()
test = test_df.copy()

y_train = train["target_declining_proxy"].astype(int)
y_test = test["target_declining_proxy"].astype(int)

if y_train.nunique() != 2 or y_test.nunique() != 2:
    raise ValueError(
        "Both train and test must contain both target classes."
    )

print("Training rows:", len(train))
print("Test rows:", len(test))
print("Test positive rate:", round(y_test.mean(), 4))

# 2. Recalculate the exact ML-07 rule score on held-out rows
# Keep the original four explicit rules; do not use the target.
def calculate_ml07_score(frame):
    out = frame.copy()

    out["low_ctr_opportunity"] = (
        (out["impressions_90d"] >= 100)
        & out["ctr"].notna()
        & (out["ctr"] < 1.0)
    )

    out["stale_content"] = (
        (out["content_age_days"] >= 180)
        & (out["days_since_last_update"] >= 90)
    )

    out["thin_content"] = (
        out["word_count"].notna()
        & (out["word_count"] < 1000)
    )

    out["missing_word_count"] = out["word_count"].isna()

    out["baseline_score"] = (
        out["low_ctr_opportunity"].astype(int)
        + out["stale_content"].astype(int)
        + out["thin_content"].astype(int)
        + out["missing_word_count"].astype(int)
    )

    return out

baseline_test = calculate_ml07_score(test)

# 3. Select features available without direct label-source fields
# IDs, trend labels, trend percentages, and recent comparison windows
# are deliberately excluded.
numeric_candidates = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

categorical_candidates = [
    "content_type",
    "main_intent",
    "competition_level",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "provider_used",
    "model_used",
]

numeric_features = [
    col for col in numeric_candidates
    if col in train.columns and col in test.columns
]

categorical_features = [
    col for col in categorical_candidates
    if col in train.columns and col in test.columns
]

feature_columns = numeric_features + categorical_features

if not feature_columns:
    raise ValueError("No eligible model features were found.")

# Treat avg_position=0 as unavailable, not as a genuine best rank.
for frame in (train, test):
    if "avg_position" in frame.columns:
        frame.loc[frame["avg_position"] == 0, "avg_position"] = np.nan

X_train = train[feature_columns].copy()
X_test = test[feature_columns].copy()

# Convert candidate numeric columns safely
for col in numeric_features:
    X_train[col] = pd.to_numeric(X_train[col], errors="coerce")
    X_test[col] = pd.to_numeric(X_test[col], errors="coerce")

# 4. Preprocess and train Logistic Regression
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(X_train, y_train)

# Probability assigned to the positive (declining-proxy) class
positive_class_index = list(
    model.named_steps["classifier"].classes_
).index(1)

model_probability = model.predict_proba(X_test)[:, positive_class_index]

# 5. Calculate both approaches on the same held-out test rows
baseline_score = baseline_test["baseline_score"].to_numpy()

# Same top-20 evaluation size for both rankings.
# Stable sorting makes ties deterministic.
baseline_order = (
    baseline_test.assign(_original_order=np.arange(len(baseline_test)))
    .sort_values(
        ["baseline_score", "impressions_90d", "content_id"],
        ascending=[False, False, True],
        kind="mergesort"
    ).index
)

baseline_ranked_positions = baseline_test.index.get_indexer(baseline_order)
baseline_top20_indices = baseline_ranked_positions[:20]

model_order = np.argsort(-model_probability, kind="stable")
model_top20_indices = model_order[:20]

def precision_at_k(y_true, ranked_indices, k=20):
    k = min(k, len(ranked_indices))
    if k == 0:
        return np.nan
    return float(np.asarray(y_true)[ranked_indices[:k]].mean())

results = []

# ML-07 rule-based score
results.append({
    "Approach": "ML-07 rule-based baseline",
    "Average Precision": average_precision_score(
        y_test, baseline_score
    ),
    "ROC-AUC": roc_auc_score(y_test, baseline_score),
    "Precision@20": precision_at_k(
        y_test.to_numpy(), baseline_top20_indices, 20
    )
})

# ML-08 Logistic Regression
results.append({
    "Approach": "ML-08 Logistic Regression",
    "Average Precision": average_precision_score(
        y_test, model_probability
    ),
    "ROC-AUC": roc_auc_score(y_test, model_probability),
    "Precision@20": precision_at_k(
        y_test.to_numpy(), model_order, 20
    )
})

results_df = pd.DataFrame(results)

print("\n--- COMPARISON ON THE SAME HELD-OUT TEST SET ---")
display(results_df.round(4))

print("Test positive rate:", round(float(y_test.mean()), 4))
print("Features used:", feature_columns)
print("\nLeakage-sensitive fields excluded:")
print([
    "trend_direction", "trend_pct",
    "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d",
    "sessions_last_30d", "sessions_prev_30d",
    "content_id", "client_id"
])

# Save in memory for Section 4; do not commit generated CSV files.
section3_predictions = pd.DataFrame({
    "content_id": test["content_id"].to_numpy(),
    "client_id": test["client_id"].to_numpy(),
    "y_true": y_test.to_numpy(),
    "baseline_score": baseline_score,
    "model_probability": model_probability,
    "model_prediction": (model_probability >= 0.5).astype(int),
})

assert len(section3_predictions) == len(test)
assert section3_predictions["content_id"].is_unique

print("\nPASS: both approaches evaluated on the same test rows.")
print("PASS: prediction table created for Section 4 error analysis.")


Training rows: 23837
Test rows: 6163
Test positive rate: 0.511

--- COMPARISON ON THE SAME HELD-OUT TEST SET ---


,Approach,Average Precision,ROC-AUC,Precision@20
0,ML-07 rule-based baseline,0.5124,0.5048,0.35
1,ML-08 Logistic Regression,0.5790,0.5841,0.85


Test positive rate: 0.511
Features used: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'content_type', 'main_intent', 'competition_level', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'provider_used', 'model_used']

Leakage-sensitive fields excluded:
['trend_direction', 'trend_pct', 'impressions_last_30d', 'impressions_prev_30d', 'clicks_last_30d', 'clicks_prev_30d', 'sessions_last_30d', 'sessions_prev_30d', 'content_id', 'client_id']

PASS: both approaches evaluated on the same test rows.
PASS: prediction table created for Section 4 error analysis.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The initial held-out results show higher Average Precision, ROC-AUC, and Precision@20 for Logistic Regression than for the ML-07 rule-based baseline. These metrics measure agreement with the existing declining-trend proxy, not future decline. I will inspect false positives and false negatives, the confusion matrix, and the strongest model coefficients to understand likely errors and influential features. The dataset contains systematic missing word counts for keyword articles, and correlated performance features may reflect the same observation period as the proxy label. Results are directional and should support human review only; independent future-period validation is needed before operational use.

In [4]:

# ============================================================
# ML-08 | SECTION 4: ERRORS AND INTERPRETATION
# Confusion matrix, false positives/negatives, model coefficients
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score
)

# 1. Confirm Section 3 ran successfully
if "section3_predictions" not in globals():
    raise RuntimeError("Run Section 3 first.")

if "model" not in globals():
    raise RuntimeError("Trained Logistic Regression model not found.")

pred = section3_predictions.copy()

required = {
    "y_true", "model_probability",
    "model_prediction", "content_id"
}
missing = required - set(pred.columns)

if missing:
    raise ValueError(f"Missing prediction fields: {sorted(missing)}")

y_true = pred["y_true"].astype(int)
y_pred = pred["model_prediction"].astype(int)

# 2. Confusion matrix and classification report
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

cm_df = pd.DataFrame(
    cm,
    index=["Actual non-down proxy", "Actual down proxy"],
    columns=["Predicted non-down", "Predicted down"]
)

print("--- CONFUSION MATRIX ---")
display(cm_df)

print("\n--- CLASSIFICATION REPORT ---")
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1],
    target_names=["Non-down proxy", "Down proxy"],
    zero_division=0
))

# 3. Inspect false positives and false negatives
pred["error_type"] = "Correct"
pred.loc[(y_true == 0) & (y_pred == 1), "error_type"] = "False positive"
pred.loc[(y_true == 1) & (y_pred == 0), "error_type"] = "False negative"

print("\nError counts:")
display(pred["error_type"].value_counts())

# Join descriptive features from the held-out test set only
inspection_cols = [
    col for col in [
        "content_id", "content_type", "main_intent",
        "impressions_90d", "ctr", "avg_position",
        "content_age_days", "days_since_last_update",
        "word_count", "engagement_rate", "scroll_rate",
        "ai_traffic_pct"
    ]
    if col in test_df.columns
]

inspection = test_df[inspection_cols].copy()
inspection = inspection.drop_duplicates(subset=["content_id"])

error_table = pred.merge(
    inspection,
    on="content_id",
    how="left",
    validate="one_to_one"
)

error_columns = [
    col for col in [
        "content_id", "error_type", "y_true",
        "model_prediction", "model_probability",
        "content_type", "impressions_90d", "ctr",
        "avg_position", "content_age_days",
        "days_since_last_update", "word_count"
    ]
    if col in error_table.columns
]

print("\n--- FALSE POSITIVES: predicted down, actual label is not down ---")
display(
    error_table[error_table["error_type"] == "False positive"]
    .sort_values("model_probability", ascending=False)
    [error_columns].head(10)
)

print("\n--- FALSE NEGATIVES: actual down label, model predicted otherwise ---")
display(
    error_table[error_table["error_type"] == "False negative"]
    .sort_values("model_probability", ascending=True)
    [error_columns].head(10)
)

# 4. Examine the strongest Logistic Regression coefficients
# Positive coefficients raise the model's score for the down proxy;
# negative coefficients lower it, conditional on other features.
try:
    feature_names = model.named_steps["preprocessor"].get_feature_names_out()
    coefficients = model.named_steps["classifier"].coef_[0]

    coef_df = pd.DataFrame({
        "feature": feature_names,
        "coefficient": coefficients
    })

    print("\n--- STRONGEST POSITIVE COEFFICIENTS ---")
    display(
        coef_df.sort_values("coefficient", ascending=False)
        .head(10)
        .round(4)
    )

    print("\n--- STRONGEST NEGATIVE COEFFICIENTS ---")
    display(
        coef_df.sort_values("coefficient", ascending=True)
        .head(10)
        .round(4)
    )

except Exception as exc:
    print("Could not extract coefficients:", repr(exc))
    coef_df = pd.DataFrame()

# 5. Check whether missing word count is concentrated by content type
if "content_type" in test_df.columns and "word_count" in test_df.columns:
    missingness = (
        test_df.assign(word_count_missing=test_df["word_count"].isna())
        .groupby("content_type", dropna=False)["word_count_missing"]
        .agg(rows="size", missing_rows="sum", missing_rate="mean")
        .sort_values("missing_rate", ascending=False)
    )

    print("\n--- WORD-COUNT MISSINGNESS IN TEST SET ---")
    display(missingness.round(3))

# 6. Concise, data-backed summary
tn, fp, fn, tp = cm.ravel()

print("\n--- SUMMARY ---")
print("True positives:", int(tp))
print("False positives:", int(fp))
print("True negatives:", int(tn))
print("False negatives:", int(fn))
print("Precision at threshold 0.5:", round(
    precision_score(y_true, y_pred, zero_division=0), 4
))
print("Recall at threshold 0.5:", round(
    recall_score(y_true, y_pred, zero_division=0), 4
))
print(
    "\nInterpretation: these errors refer to the existing trend proxy. "
    "They do not establish future decline, causal refresh impact, "
    "or readiness for production."
)


--- CONFUSION MATRIX ---


,Predicted non-down,Predicted down
Actual non-down proxy,1393,1621
Actual down proxy,1057,2092



--- CLASSIFICATION REPORT ---
                precision    recall  f1-score   support

Non-down proxy       0.57      0.46      0.51      3014
    Down proxy       0.56      0.66      0.61      3149

      accuracy                           0.57      6163
     macro avg       0.57      0.56      0.56      6163
  weighted avg       0.57      0.57      0.56      6163


Error counts:


,count
error_type,
Correct,3485
False positive,1621
False negative,1057



--- FALSE POSITIVES: predicted down, actual label is not down ---


...


--- STRONGEST NEGATIVE COEFFICIENTS ---


,feature,coefficient
27,numeric__missingindicator_avg_position,-1.0229
9,numeric__users_90d,-1.0152
61,categorical__model_used_unknown,-0.6059
45,categorical__freshness_tier_31-90,-0.5885
14,numeric__days_with_sessions,-0.4368
34,categorical__main_intent_navigational,-0.3992
29,categorical__content_type_comparison article,-0.3575
15,numeric__content_age_days,-0.2725
18,numeric__avg_position,-0.2469
49,categorical__word_count_tier_3500+,-0.2436



--- WORD-COUNT MISSINGNESS IN TEST SET ---


,rows,missing_rows,missing_rate
content_type,,,
keyword article,6163,1085,0.176



--- SUMMARY ---
True positives: 2092
False positives: 1621
True negatives: 1393
False negatives: 1057
Precision at threshold 0.5: 0.5634
Recall at threshold 0.5: 0.6643

Interpretation: these errors refer to the existing trend proxy. They do not establish future decline, causal refresh impact, or readiness for production.


In [5]:
print("TRAIN content types:")
display(train_df["content_type"].value_counts(dropna=False))

print("\nTEST content types:")
display(test_df["content_type"].value_counts(dropna=False))

print("\nUnique train clients:", train_df["client_id"].nunique())
print("Unique test clients:", test_df["client_id"].nunique())
print(
    "Client overlap:",
    len(set(train_df["client_id"]) & set(test_df["client_id"]))
)

TRAIN content types:


,count
content_type,
keyword article,21044
feedly article,2096
comparison article,697



TEST content types:


,count
content_type,
keyword article,6163



Unique train clients: 25
Unique test clients: 7
Client overlap: 0


### Validation limitation: content-type distribution

The client-grouped split held out 7 clients and showed no client overlap between training and test sets. However, the test set contains only keyword articles (6,163 rows), while the training set also contains feedly and comparison articles. Therefore, the reported metrics evaluate performance on keyword articles from held-out clients only. They should not be generalized to other content types. Future validation should examine content-type coverage and use a split design that supports the intended deployment population, while preserving client separation and preventing temporal leakage.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.